# model_ml_001  –  KIER ML 예측 모델 (CB / XGB / LGBM / DT / RF)
`ML-01 단일` · `ML-02 KFold CV` · `ML-03 군집화 CV` · `ML-04 비교` 통합

## 01. Init

In [ ]:
#region Basic_Import
## Basic
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
pd.options.display.float_format = '{:.10f}'.format

## Datetime
from datetime import datetime, date, timedelta

## glob
import glob, requests, json

## 시각화
import matplotlib.pyplot as plt, seaborn as sns
# %matplotlib inline
plt.rcParams['figure.figsize'] = [10, 8]

from scipy import stats

## Split, 정규화
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# K-Means 알고리즘
from sklearn.cluster import KMeans, MiniBatchKMeans

# Clustering 알고리즘의 성능 평가 측도
from sklearn import metrics
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score, adjusted_rand_score, silhouette_score, rand_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.cluster import contingency_matrix

## Modeling, Model Training
from sklearn.model_selection import train_test_split, KFold, GridSearchCV

## Grid Search
# kfold = KFold(n_splits = 5, shuffle = False, random_state = None)

## For Web
#endregion Basic_Import

In [ ]:
## ML 라이브러리
# !pip install catboost lightgbm
from catboost import Pool, CatBoostRegressor
import lightgbm as lgbm
from lightgbm import LGBMRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [ ]:
## Import_Local
from core import model_ml          as com_Model
from core import data_datetime     as com_date
from core import data_analysis     as com_Analysis
from core import provider_kier_m02 as com_KIER_M02
from core import provider_kma      as com_KMA
from core import data_clustering   as com_clustering

In [ ]:
## Init_config
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

In [ ]:
## Define Todate str
str_now_ymd = dt.datetime.now().date()
str_now_y, str_now_m, str_now_d = dt.datetime.now().year, dt.datetime.now().month, dt.datetime.now().day
str_now_hr, str_now_min = dt.datetime.now().hour, dt.datetime.now().minute

print(dt.datetime.now())
print(str(str_now_y) + " / " + str(str_now_m)  + " / " + str(str_now_d))
print(str(str_now_hr) + " : " + str(str_now_min))

In [ ]:
## Dict_Domain
int_domain   = 0   # 0=ELEC 1=HEAT 2=WATER 3=HOT_HEAT 4=HOT_FLOW 5=GAS
int_interval = 1   # 0=10min 1=1H 2=1D 3=1W 4=1M
K            = 3   # 군집 수 (clustering_001에서 결정)
int_grp      = 0   # 0=ALL 1=C0 2=C1 3=C2

str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
_, _, _, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)
str_file  = 'KIER_' + str_domain + '_INST_1H_Resampled.csv'
str_interval_str, *_ = com_KIER_M02.create_file_str(str_domain, int_interval)

## 02. 공통 함수 정의
군집화 로더 / 데이터셋 준비 함수

In [ ]:
def cluster_label():
    df_kier_raw = pd.read_csv(str_dirName_h + str_file, index_col = 0)
    df_kier_raw['METER_DATE'] = pd.to_datetime(df_kier_raw['METER_DATE'])

    ## 호실별 순시 사용량 컬럼만 가져오기
    list_col_tar = list(df_kier_raw.columns[1:])
    df_kier_h = df_kier_raw.set_index('METER_DATE')

    # ## Error Log : "[5:-2]" 부분을 추가하여 연월일시 및 평균합계 부분을 제거해주지 않으면, 군집화 계수가 제대로 도출되지 못함.
    # df_kier_summary_total = df_kier_h.transpose().reset_index()[5:-2]
    # ## 또는, 가장 깔끔하게 이렇게 처리해도 좋다
    df_kier_summary_total = df_kier_h[list_col_tar].transpose().reset_index()

    ## 세대 번호의 컬럼명이 'index'로 지정되어 오류 발생
    df_kier_summary_total['h_index'] = df_kier_summary_total['index']
    df_kier_summary_total = df_kier_summary_total.drop(columns = ['index'])

    X = df_kier_summary_total.drop(columns = 'h_index')
    y = df_kier_summary_total['h_index']

    # 변수 표준화
    scaler = StandardScaler() # 변수 표준화 클래스
    scaler.fit(X)  # 표준화를 위해 변수별 파라미터(평균, 표준편차) 계산
    X_std = scaler.transform(X)  # 훈련자료 표준화 변환

    ## 최종 군집에 대한 Labeled Data 저장
    km = KMeans(n_clusters = K, init="k-means++", max_iter=300, n_init=1).fit(X_std)
    list_size_cluster = com_clustering.get_cluster_sizes(km, X_std) ## 최종 군집화에 대한 군집 크기
    df_kier_summary_total['target_'+str_domain] = 0
    for i in range(0, len(df_kier_summary_total)) : df_kier_summary_total['target_'+str_domain].iloc[i] = km.labels_[i]

    str_file_labeled = str_dirName_h + 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '_K' + str(K) + '.csv'
    df_kier_summary_total = df_kier_summary_total[['h_index', 'target_'+str_domain]]
    df_kier_summary_total.to_csv(str_file_labeled)

    return df_kier_summary_total, list_size_cluster

In [ ]:
def load_dataset_Not_cluster():
    ## ▶ Dataset 불러오기
    ## 1. Interpolate / Filled ASOS Data
    str_file = '../data_Energy_KIER/KMA_ASOS_119_2010_2023_1st_to CSV.csv'
    df_ASOS = pd.read_csv(str_file, index_col = 0).reset_index()

    try : df_ASOS['METER_DATE'] = pd.to_datetime(df_ASOS['METER_DATE'])
    except KeyError : df_kier_raw = com_date.create_col_datetime(df_ASOS, 'METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE').drop(labels = ['None'], axis = 1)

    ## 3. 1시간 단위 사용량 Data Load
    str_file = 'KIER_' + str_domain + '_INST_1H_Resampled.csv'
    df_raw = pd.read_csv(str_dirName_h + str_file, index_col = 0)

    ## ▶ h_index에 따라 Dataset 분리
    ## 1. 각 index별 house 목록 생성
    list_kier_h_all = df_kier_h_cluster['h_index']

    ## 2. 전체 사용량 합계 구하기
    df_kier_h_all = df_raw.copy()
    df_kier_h_all['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
    df_kier_h_tmp = df_raw[list_kier_h_all]
    df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_tmp.sum(axis = 1)
    ## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
    df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_all[str_domain + '_INST_SUM_ALL'].shift(1)
    df_kier_h_all.dropna()

    ## 4. 날씨 데이터 추가
    df_kier_h_all = pd.merge(df_kier_h_all, df_ASOS, how = 'left', on = ['METER_DATE'])
    df_kier_h_all = com_KMA.Interpolate_KMA_ASOS(df_kier_h_all)
    df_kier_h_all = com_date.create_col_ymdhm(df_kier_h_all, 'METER_DATE')

    # str_col_tar = str_domain + '_INST_SUM_' + dict_grp[int_grp]
    str_col_tar = str_domain + '_INST_SUM_ALL'
    df_tar_res = df_kier_h_all.drop(columns = ['METER_DATE', 'DAY']).dropna()

    return df_tar_res, str_col_tar

In [ ]:
def load_dataset_cluster(int_grp):
    ## ▶ Dataset 불러오기
    ## 1. Interpolate / Filled ASOS Data
    str_file = '../data_Energy_KIER/KMA_ASOS_119_2010_2023_1st_to CSV.csv'
    df_ASOS = pd.read_csv(str_file, index_col = 0).reset_index()

    try : df_ASOS['METER_DATE'] = pd.to_datetime(df_ASOS['METER_DATE'])
    except KeyError : df_kier_raw = com_date.create_col_datetime(df_ASOS, 'METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE').drop(labels = ['None'], axis = 1)

    ## 2. Labeled Data Load
    ## Cluster 기준 Interval
    str_file_clustering = 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '_K' + str(K) + '.csv'
    df_kier_h_cluster = pd.read_csv(str_dirName_h + str_file_clustering
                                    , index_col = 0).rename(columns = {'index' : 'h_index'})[['h_index', 'target_' + str_domain]]
    # print(str_interval)
    # print(df_kier_h_cluster['target_' + str_domain].drop_duplicates())
    # df_kier_h_cluster

    ## 3. 1시간 단위 사용량 Data Load
    str_file = 'KIER_' + str_domain + '_INST_1H_Resampled.csv'
    df_raw = pd.read_csv(str_dirName_h + str_file, index_col = 0)



    ## ▶ h_index에 따라 Dataset 분리
    ## 1. 각 index별 house 목록 생성
    list_kier_h_all = df_kier_h_cluster['h_index']
    # print(len(list_kier_h_all))
    list_kier_h_c0 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 0]['h_index']
    # print(len(list_kier_h_c0))
    list_kier_h_c1 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 1]['h_index']
    # print(len(list_kier_h_c1))

    if K == 3 : list_kier_h_c2 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 2]['h_index']
    # print(len(list_kier_h_c2))

    ## 2. 전체 사용량 합계 구하기
    df_kier_h_all = df_raw.copy()
    df_kier_h_all['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
    df_kier_h_tmp = df_raw[list_kier_h_all]
    df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_tmp.sum(axis = 1)
    ## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
    df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_all[str_domain + '_INST_SUM_ALL'].shift(1)
    df_kier_h_all.dropna()

    ## 3. Cluster별 사용량 합계 산출
    ## ■ C00
    df_kier_h_c0 = df_raw.copy()[list_kier_h_c0]
    df_kier_h_c0['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
    df_kier_h_tmp = df_raw[list_kier_h_c0]
    df_kier_h_c0[str_domain + '_INST_SUM_C0'] = df_kier_h_tmp.sum(axis = 1)
    ## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
    df_kier_h_c0[str_domain + '_INST_SUM_C0'] = df_kier_h_c0[str_domain + '_INST_SUM_C0'].shift(1)
    df_kier_h_c0.dropna()

    ## ■ C01
    df_kier_h_c1 = df_raw.copy()[list_kier_h_c1]
    df_kier_h_c1['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
    df_kier_h_tmp = df_raw[list_kier_h_c1]
    df_kier_h_c1[str_domain + '_INST_SUM_C1'] = df_kier_h_tmp.sum(axis = 1)
    ## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
    df_kier_h_c1[str_domain + '_INST_SUM_C1'] = df_kier_h_c1[str_domain + '_INST_SUM_C1'].shift(1)
    df_kier_h_c1.dropna()

    if K == 3:
        ## ■ C02
        df_kier_h_c2 = df_raw.copy()[list_kier_h_c2]
        df_kier_h_c2['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
        df_kier_h_tmp = df_raw[list_kier_h_c2]
        df_kier_h_c2[str_domain + '_INST_SUM_C2'] = df_kier_h_tmp.sum(axis = 1)
        ## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
        df_kier_h_c2[str_domain + '_INST_SUM_C2'] = df_kier_h_c2[str_domain + '_INST_SUM_C2'].shift(1)
        df_kier_h_c2.dropna()

    ## 4. 날씨 데이터 추가
    df_kier_h_all = pd.merge(df_kier_h_all, df_ASOS, how = 'left', on = ['METER_DATE'])
    df_kier_h_all = com_KMA.Interpolate_KMA_ASOS(df_kier_h_all)
    df_kier_h_all = com_date.create_col_ymdhm(df_kier_h_all, 'METER_DATE')

    df_kier_h_c0 = pd.merge(df_kier_h_c0, df_ASOS, how = 'left', on = ['METER_DATE'])
    df_kier_h_c0 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c0)
    df_kier_h_c0 = com_date.create_col_ymdhm(df_kier_h_c0, 'METER_DATE')

    df_kier_h_c1 = pd.merge(df_kier_h_c1, df_ASOS, how = 'left', on = ['METER_DATE'])
    df_kier_h_c1 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c1)
    df_kier_h_c1 = com_date.create_col_ymdhm(df_kier_h_c1, 'METER_DATE')

    if K == 3:
        df_kier_h_c2 = pd.merge(df_kier_h_c2, df_ASOS, how = 'left', on = ['METER_DATE'])
        df_kier_h_c2 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c2)
        df_kier_h_c2 = com_date.create_col_ymdhm(df_kier_h_c2, 'METER_DATE')

    ## 모든 세대
    if int_grp == 0 : df_tar_res = df_kier_h_all.drop(columns = ['METER_DATE', 'DAY']).dropna()
    ## 군집 C0
    elif int_grp == 1 : df_tar_res = df_kier_h_c0.drop(columns = ['METER_DATE', 'DAY']).dropna()
    ## 군집 C1
    elif int_grp == 2 : df_tar_res = df_kier_h_c1.drop(columns = ['METER_DATE', 'DAY']).dropna()
    ## 군집 C0
    elif int_grp == 3 : df_tar_res = df_kier_h_c2.drop(columns = ['METER_DATE', 'DAY']).dropna()

    str_col_tar = str_domain + '_INST_SUM_' + dict_grp[int_grp]

    return df_tar_res, str_col_tar

## 03. ML 단일 실행
`ML-01` 기준 · CB / XGB / LGBM / DT / RF 순차 실행

In [ ]:
## KMA_ASOS Data
# str_dir_kmaAsos = "../data/data_KMA_ASOS/"

## Interpolate / Filled ASOS Data
str_file = '../data_Energy_KIER/KMA_ASOS_119_2010_2023_1st_to CSV.csv'
df_ASOS = pd.read_csv(str_file, index_col = 0).reset_index()

try : df_ASOS['METER_DATE'] = pd.to_datetime(df_ASOS['METER_DATE'])
except KeyError : df_kier_raw = com_date.create_col_datetime(df_ASOS, 'METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE').drop(labels = ['None'], axis = 1)

df_ASOS

In [ ]:
## Cluster 기준 Interval
str_file_clustering = 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '_K' + str(K) + '.csv'
df_kier_h_cluster = pd.read_csv(str_dirName_h + str_file_clustering
                                , index_col = 0).rename(columns = {'index' : 'h_index'})[['h_index', 'target_' + str_domain]]
print(str_interval)
print(df_kier_h_cluster['target_' + str_domain].drop_duplicates())
# df_kier_h_cluster

In [ ]:
list_kier_h_all = df_kier_h_cluster['h_index']
print(len(list_kier_h_all))
list_kier_h_c0 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 0]['h_index']
print(len(list_kier_h_c0))
list_kier_h_c1 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 1]['h_index']
print(len(list_kier_h_c1))

if K == 3 : 
    list_kier_h_c2 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 2]['h_index']
    print(len(list_kier_h_c2))

In [ ]:
## 사용량 Data Load
## 1시간 단위
str_file = 'KIER_' + str_domain + '_INST_1H_Resampled.csv'
df_raw = pd.read_csv(str_dirName_h + str_file, index_col = 0)
df_raw

In [ ]:
## 전체 사용량 합계
df_kier_h_all = df_raw.copy()
df_kier_h_all['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_tmp = df_raw[list_kier_h_all]
df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_tmp.sum(axis = 1)
## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_kier_h_all[str_domain + '_INST_SUM_ALL'].shift(1)
df_kier_h_all.dropna()

In [ ]:
## Cluster별 사용량 합계
## ■ C00
df_kier_h_c0 = df_raw.copy()[list_kier_h_c0]
df_kier_h_c0['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_tmp = df_raw[list_kier_h_c0]
df_kier_h_c0[str_domain + '_INST_SUM_C0'] = df_kier_h_tmp.sum(axis = 1)
## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
df_kier_h_c0[str_domain + '_INST_SUM_C0'] = df_kier_h_c0[str_domain + '_INST_SUM_C0'].shift(1)
df_kier_h_c0.dropna()

## ■ C01
df_kier_h_c1 = df_raw.copy()[list_kier_h_c1]
df_kier_h_c1['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_tmp = df_raw[list_kier_h_c1]
df_kier_h_c1[str_domain + '_INST_SUM_C1'] = df_kier_h_tmp.sum(axis = 1)
## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
df_kier_h_c1[str_domain + '_INST_SUM_C1'] = df_kier_h_c1[str_domain + '_INST_SUM_C1'].shift(1)
df_kier_h_c1.dropna()

if K == 3:
    ## ■ C02
    df_kier_h_c2 = df_raw.copy()[list_kier_h_c2]
    df_kier_h_c2['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
    df_kier_h_tmp = df_raw[list_kier_h_c2]
    df_kier_h_c2[str_domain + '_INST_SUM_C2'] = df_kier_h_tmp.sum(axis = 1)
    ## 시점을 밀어서, 세대별 사용량을 과거 사용량으로 사용
    df_kier_h_c2[str_domain + '_INST_SUM_C2'] = df_kier_h_c2[str_domain + '_INST_SUM_C2'].shift(1)
    df_kier_h_c2.dropna()

In [ ]:
# 에너지 사용량 총계만
# df_kier_h_all = df_kier_h_all[['METER_DATE', str_domain + '_INST_SUM_ALL']].dropna()

# df_kier_h_c0 = df_kier_h_c0[['METER_DATE', str_domain + '_INST_SUM_C0']].dropna()

# df_kier_h_c1 = df_kier_h_c1[['METER_DATE', str_domain + '_INST_SUM_C1']].dropna()

# df_kier_h_c2 = df_kier_h_c2[['METER_DATE', str_domain + '_INST_SUM_C2']].dropna()


In [ ]:
## 날씨 데이터 추가
df_kier_h_all = pd.merge(df_kier_h_all, df_ASOS, how = 'left', on = ['METER_DATE'])
df_kier_h_all = com_KMA.Interpolate_KMA_ASOS(df_kier_h_all)
df_kier_h_all = com_date.create_col_ymdhm(df_kier_h_all, 'METER_DATE')

df_kier_h_c0 = pd.merge(df_kier_h_c0, df_ASOS, how = 'left', on = ['METER_DATE'])
df_kier_h_c0 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c0)
df_kier_h_c0 = com_date.create_col_ymdhm(df_kier_h_c0, 'METER_DATE')

df_kier_h_c1 = pd.merge(df_kier_h_c1, df_ASOS, how = 'left', on = ['METER_DATE'])
df_kier_h_c1 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c1)
df_kier_h_c1 = com_date.create_col_ymdhm(df_kier_h_c1, 'METER_DATE')

if K == 3:
    df_kier_h_c2 = pd.merge(df_kier_h_c2, df_ASOS, how = 'left', on = ['METER_DATE'])
    df_kier_h_c2 = com_KMA.Interpolate_KMA_ASOS(df_kier_h_c2)
    df_kier_h_c2 = com_date.create_col_ymdhm(df_kier_h_c2, 'METER_DATE')

In [ ]:
pd.set_option('display.max_row', 500)

print(df_kier_h_all.shape)
print(df_kier_h_all.columns)
print(df_kier_h_all.isna().sum())
print(df_kier_h_c0.shape)
print(df_kier_h_c0.columns)
print(df_kier_h_c0.isna().sum())
print(df_kier_h_c1.shape)
print(df_kier_h_c1.columns)
print(df_kier_h_c1.isna().sum())

if K == 3:
    print(df_kier_h_c2.shape)
    print(df_kier_h_c2.columns)
    print(df_kier_h_c2.isna().sum())

### 세대 선택 및 모델 실행

In [ ]:
## 모든 세대
if int_grp == 0 : df_tar = df_kier_h_all
## 군집 C0
elif int_grp == 1 : df_tar = df_kier_h_c0
## 군집 C1
elif int_grp == 2 : df_tar = df_kier_h_c1
## 군집 C0
elif int_grp == 3 : df_tar = df_kier_h_c2
str_col_tar = str_domain + '_INST_SUM_' + dict_grp[int_grp]

# df_tar = com_date.create_col_ymdhm(df_tar, 'METER_DATE')
# df_tar = com_date.create_col_weekdays(df_tar, 'METER_DATE')
df_tar = df_tar.drop(columns = ['METER_DATE', 'DAY']).dropna()
# df_tar = df_tar.drop(columns = ['METER_DATE', 'day_of_the_week']).dropna()
# df_tar = df_tar.drop(columns = ['METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'None']).dropna()

# trainSet_Origin, testSet_Origin = train_test_split(df_tar, test_size = 0.3, shuffle = False)
# print(trainSet_Origin.shape, testSet_Origin.shape)

In [ ]:
print(str_domain, ' /// ', str_interval, ' /// ', str_col_tar)

dict_model = {0 : 'CB', 1 : 'DT', 2 : 'LGBM', 3 : 'RF', 4 : 'XGB'}

list_result = [] ## 조금 원시적인 방법이나, 추후 개선 예정
for int_model in range(0, 5):
    list_kf_mae, list_kf_mape, list_kf_mse, list_kf_rmse, list_kf_msle, list_kf_mbe, list_kf_r2, list_kf_tm_code = [], [], [], [], [], [], [], []
    list_kf_scores = [] ## 최종 출력될 Score List

    d_actual, model_preds, tm_code = com_Model.model_ml_analysis_single(df_tar, int_model, 0.3, str_col_tar)
    list_scores = com_Model.model_sk_metrics(d_actual, model_preds)
    list_scores.append(tm_code)

    str_title = str_domain + ' by ' + dict_model[int_model] + ' (Interval - ' + str_interval + ', grp - ' + dict_grp[int_grp] + ') '
    str_path = '..//Result_Model_2024-08/' + str_title + '.png'
    com_Model.model_visualization(d_actual, model_preds, str_title, str_path = str_path)
    # print("▶ " + dict_model[int_model])
    # print(list_scores)

    list_result.append(dict_model[int_model])
    list_result.append(list_scores)

In [ ]:
str_title = str_domain + ' by ML (Interval - ' + str_interval + ', grp - ' + dict_grp[int_grp] + ') '
str_path = '..//Result_Model_2024-08/' + str_title + '.txt'

with open(str_path, "w") as file_txt:
    file_txt.write(str(list_result))

## 04. ML KFold / TimeSeriesSplit CV
`ML-02` 기준

In [ ]:
## 모든 세대
if int_grp == 0 : df_tar = df_kier_h_all
## 군집 C0
elif int_grp == 1 : df_tar = df_kier_h_c0
## 군집 C1
elif int_grp == 2 : df_tar = df_kier_h_c1
## 군집 C0
elif int_grp == 3 : df_tar = df_kier_h_c2
str_col_tar = str_domain + '_INST_SUM_' + dict_grp[int_grp]

# df_tar = com_date.create_col_ymdhm(df_tar, 'METER_DATE')
# df_tar = com_date.create_col_weekdays(df_tar, 'METER_DATE')
df_tar = df_tar.drop(columns = ['METER_DATE', 'DAY']).dropna()
# df_tar = df_tar.drop(columns = ['METER_DATE', 'day_of_the_week']).dropna()
# df_tar = df_tar.drop(columns = ['METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'None']).dropna()

# trainSet_Origin, testSet_Origin = train_test_split(df_tar, test_size = 0.3, shuffle = False)
# print(trainSet_Origin.shape, testSet_Origin.shape)
df_tar

In [ ]:
# list_kf_scores = com_Model.model_ml_analysis_with_KFold(df_tar, 0, 0.3, str_col_tar, 5)
# list_kf_scores

In [ ]:
from sklearn.model_selection import KFold, TimeSeriesSplit 

dict_model = {0 : 'CB', 1 : 'DT', 2 : 'LGBM', 3 : 'RF', 4 : 'XGB'}
float_rate = 0.3
test_size = round(len(df_tar) * float_rate)
int_fold = 10

for int_model in range(0, 5) : 
    list_res, list_hists = com_Model.model_ml_analysis_with_KFold(df_tar, int_model, float_rate, str_col_tar, int_fold)

    ## list_res 저장
    str_txt = "../kf_result/kf_result_" + dict_model[int_model] + '_K' + str(int_fold) + '.txt'
    file_txt = open(str_txt, 'w')
    print(list_res, file = file_txt)

    ## list_hist 저장
    str_txt = "../kf_hist/kf_hist_" + dict_model[int_model] + '_K' + str(int_fold) + '.txt'
    file_txt = open(str_txt, 'w')
    print(list_hists, file = file_txt)

    ## open 후 다른 것을 open하면 자동으로 close되어 저장되지만,
    ## 마지막 파일은 반드시 close를 통해 종료해야만 저장이 완료됨
    file_txt.close()


In [ ]:
list_hists

In [ ]:
list_res

## 05. ML + 군집화 KFold CV
`ML-03` 기준 · cluster_label() 사용

In [ ]:
str_file_asos = '../data_Energy_KIER/KMA_ASOS_119_2010_2023_1st_to CSV.csv'  ## ▶ ASOS 파일 경로

import sys
from sklearn.model_selection import KFold, TimeSeriesSplit 
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

float_rate = 0.3
int_fold = 10

## Dict_Domain
## {0:"ELEC", 1:"HEAT", 2:"WATER", 3:"HOT_HEAT", 4:"HOT_FLOW", 99:"GAS"}
## K : 2 or 3
## {0 : '10MIN', 1 : '1H', 2 : '1D', 3 : '1W', 4 : '1M'}
## {0 : 'ALL', 1 : 'C0', 2 : 'C1', 3 : 'C2'}
dict_model = {0 : 'CB', 1 : 'DT', 2 : 'LGBM', 3 : 'RF', 4 : 'XGB'}
dict_interval = {0 : '10MIN', 1 : '1H', 2 : '1D', 3 : '1W', 4 : '1M'}
dict_grp = {0 : 'ALL', 1 : 'C0', 2 : 'C1', 3 : 'C2'}
int_domain, int_grp = 0, 1

K = 3 ## 2, 3
int_interval = 4 ## 3, 4
int_model = 4 ## 0, 1, 2, 3, 4

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)
## Interval, Target File
str_interval, str_fileRaw, str_fileRaw_hList, str_file = com_KIER_M02.create_file_str(str_domain, int_interval)

# print(str(os.listdir(str_dirData)) + "\n")
# print(os.listdir(str_dirName_h))

str_file_clustering = 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '_K' + str(K) + '.csv'
df_kier_h_cluster = pd.read_csv(str_dirName_h + str_file_clustering
                                , index_col = 0).rename(columns = {'index' : 'h_index'})[['h_index', 'target_' + str_domain]]
df_kier_h_cluster

In [ ]:
## ▶ [Step 1] ASOS 데이터 로드
df_ASOS = pd.read_csv(str_file_asos, index_col=0).reset_index()
try:
    df_ASOS['METER_DATE'] = pd.to_datetime(df_ASOS['METER_DATE'])
except KeyError:
    df_ASOS = com_date.create_col_datetime(
        df_ASOS, 'METER_DATE', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'MINUTE'
    ).drop(labels=['None'], axis=1)

In [ ]:
## ▶ [Step 2] KIER 1H 순시 사용량 데이터 로드
str_file_1h = 'KIER_' + str_domain + '_INST_1H_Resampled.csv'
df_raw = pd.read_csv(str_dirName_h + str_file_1h, index_col=0)

In [ ]:
## ▶ [Step 3] 전체 세대 합계 산출 및 시점 이동 (1-step lag)
list_kier_h_all = df_kier_h_cluster['h_index']
df_kier_h_all = df_raw.copy()
df_kier_h_all['METER_DATE'] = pd.to_datetime(df_kier_h_all['METER_DATE'])
df_kier_h_all[str_domain + '_INST_SUM_ALL'] = df_raw[list_kier_h_all].sum(axis=1).shift(1)

In [ ]:
## ▶ [Step 4] 날씨 데이터 추가 (Merge → Interpolation → 날짜 컬럼 생성)
df_kier_h_all = pd.merge(df_kier_h_all, df_ASOS, how='left', on=['METER_DATE'])
df_kier_h_all = com_KMA.Interpolate_KMA_ASOS(df_kier_h_all)
df_kier_h_all = com_date.create_col_ymdhm(df_kier_h_all, 'METER_DATE')

In [ ]:
## ▶ [Step 5] Target 컬럼 설정 및 최종 Dataset 구성
str_col_tar = str_domain + '_INST_SUM_ALL'
df_tar = df_kier_h_all.drop(columns=['METER_DATE', 'DAY']).dropna()
df_tar.shape

In [ ]:
## 비군집화 데이터셋 모델 실행
sys.stdout.flush()
list_res, list_hists = com_ML.model_ml_analysis_with_KFold(df_tar, int_model, float_rate, str_col_tar, int_fold, shuffle=True)

## list_res 저장
str_txt = '../kf_result_include_Clustering_' + dict_model[int_model] + '/kf_result_' + str(dict_interval[int_interval]) + '_ALL_' + dict_grp[int_grp] + '_' + dict_model[int_model] + '_CV' + str(int_fold) + '.txt'
file_txt = open(str_txt, 'w')
print('- Interval = ' + dict_interval[int_interval] + '\n'
        + '- K = 0' + '\n'
        + '- grp = ALL' + '\n'
        + '- model = ' + dict_model[int_model] + '\n'
        + '- Case = ALL' + ',' + ' size_cluster = ' + str(348) + '\n'
        + '- Size = ' + str(df_tar.shape) + '\n'
        + '- Columns = ' + str(df_tar.columns) + '\n', file = file_txt)
print(list_res, file = file_txt)

## list_hist 저장
str_txt = '../kf_hist_include_Clustering_' + dict_model[int_model] + '/kf_hist_' + str(dict_interval[int_interval]) + '_ALL_' + dict_grp[int_grp] + '_' + dict_model[int_model] + '_CV' + str(int_fold) + '.txt'
file_txt = open(str_txt, 'w')
print('- Interval = ' + dict_interval[int_interval] + '\n'
        + '- K = 0' + '\n' 
        + '- grp = ALL' + '\n'
        + '- model = ' + dict_model[int_model] + '\n'
        + '- Case = ALL' + ',' + ' size_cluster = ' + str(348) + '\n'
        + '- Size = ' + str(df_tar.shape) + '\n'
        + '- Columns = ' + str(df_tar.columns) + '\n', file = file_txt)
print(list_hists, file = file_txt)

## open 후 다른 것을 open하면 자동으로 close되어 저장되지만,
## 마지막 파일은 반드시 close를 통해 종료해야만 저장이 완료됨
file_txt.close()

## 06. 결과 비교
군집화 유무 · 군집 그룹별 예측 성능 비교 (`ML-04` 기준)

In [ ]:
import sys
from sklearn.model_selection import KFold, TimeSeriesSplit 
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

float_rate = 0.3
# test_size = round(len(df_tar) * float_rate)
int_fold = 10

## Dict_Domain
## {0:"ELEC", 1:"HEAT", 2:"WATER", 3:"HOT_HEAT", 4:"HOT_FLOW", 99:"GAS"}
## K : 2 or 3
## {0 : '10MIN', 1 : '1H', 2 : '1D', 3 : '1W', 4 : '1M'}
## {0 : 'ALL', 1 : 'C0', 2 : 'C1', 3 : 'C2'}
dict_model = {0 : 'CB', 1 : 'DT', 2 : 'LGBM', 3 : 'RF', 4 : 'XGB'}
dict_interval = {0 : '10MIN', 1 : '1H', 2 : '1D', 3 : '1W', 4 : '1M'}
dict_grp = {0 : 'ALL', 1 : 'C0', 2 : 'C1', 3 : 'C2'}
int_domain, int_grp = 0, 1

K = 2 ## 2, 3
int_interval = 3 ## 3, 4
int_model = 4 ## 0, 1, 2, 3, 4

## Domain, ACCU/INST Column
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
## Directory Root
str_dirData, str_dir_raw, str_dir_cleansed, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)
## Interval, Target File
str_interval, str_fileRaw, str_fileRaw_hList, str_file = com_KIER_M02.create_file_str(str_domain, int_interval)

# print(str(os.listdir(str_dirData)) + "\n")
# print(os.listdir(str_dirName_h))

str_file_clustering = 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '_K' + str(K) + '.csv'
df_kier_h_cluster = pd.read_csv(str_dirName_h + str_file_clustering
                                , index_col = 0).rename(columns = {'index' : 'h_index'})[['h_index', 'target_' + str_domain]]
df_kier_h_cluster

In [ ]:
## 비군집화 데이터셋에 대한 처리 (비교군)
sys.stdout.flush() ## flush
df_tar, str_col_tar = load_dataset_Not_cluster()
d_actual, model_preds, tm_code = com_ML.model_ml_analysis_single(df_tar, int_model, float_rate, str_col_tar)

## 군집화 데이터셋에 대한 처리 (실험군)
df_kier_summary_total, list_size_cluster = cluster_label()
print(list_size_cluster)
# df_kier_summary_total

for int_grp in range(1, K + 1): ## 군집 형성된 데이터셋만 분석
        print('■ ' + str(int_grp))
        df_tar, str_col_tar = load_dataset_cluster(int_grp) ## 해당 군집에 대한 데이터셋 및 Target Column
        # print(df_tar.columns)
        # print(df_tar.shape)

        if int_grp == 1 : d_actual_clust_01, model_preds_clust_01, tm_code_clust_01 = com_ML.model_ml_analysis_single(df_tar, int_model, float_rate, str_col_tar)
        elif int_grp == 2 : d_actual_clust_02, model_preds_clust_02, tm_code_clust_02 = com_ML.model_ml_analysis_single(df_tar, int_model, float_rate, str_col_tar)
        elif int_grp == 3 : d_actual_clust_03, model_preds_clust_03, tm_code_clust_03 = com_ML.model_ml_analysis_single(df_tar, int_model, float_rate, str_col_tar)

print("<ACTUAL>")
# print(list(d_actual))
# print(list(d_actual_clust_01))
# print(list(d_actual_clust_02))
# if K == 3 : print(list(d_actual_clust_03))

print("<PRED>")
# print(list(model_preds))
# print(list(model_preds_clust_01))
# print(list(model_preds_clust_02))
# if K == 3 : print(list(model_preds_clust_03))

print(len(model_preds_clust_01))
print(len(model_preds_clust_02))
if K == 3 : print(len(model_preds_clust_03))

model_preds_clust = []
for i in range(0, len(model_preds_clust_01)):
        if K == 2 : model_preds_clust.append(model_preds_clust_01[i] + model_preds_clust_02[i])
        elif K == 3 : model_preds_clust.append(model_preds_clust_01[i] + model_preds_clust_02[i] + model_preds_clust_03[i])
print(list(model_preds_clust))

In [ ]:
## 시각화 파라미터
## dpi
int_dpi = 300
## Legend 관련
if K == 2 : str_ncol, tuple_axis_allcase = 6, (0.05, -0.2)
elif K == 3 : str_ncol, tuple_axis_allcase = 4, (0.20, -0.3)
tuple_axis_2case = (0.375, -0.2)
tuple_axis_3case = (0.275, -0.2)

In [ ]:
## 대조군 : 군집화 X
## 실험군 01-01 : C0 군집의 합
## 실험군 01-02 : C1 군집의 합
## 실험군 01-03 : C2 군집의 합
## 실험군 02 : 모든 군집의 합
list_scores = com_ML.model_sk_metrics(d_actual, model_preds)
list_scores_c0 = com_ML.model_sk_metrics(d_actual_clust_01, model_preds_clust_01)
list_scores_c1 = com_ML.model_sk_metrics(d_actual_clust_02, model_preds_clust_02)
if K == 3 : list_scores_c2 = com_ML.model_sk_metrics(d_actual_clust_03, model_preds_clust_03)
list_scores_clustered = com_ML.model_sk_metrics(d_actual, model_preds_clust)

str_txt = 'ELEC_Comparison_' + str(dict_model[int_model]) + '_K' + str(K) + str(dict_interval[int_interval])[1:] + '.txt'
file_txt = open(str_txt, 'w')

print("■ 군집크기", file = file_txt)
print(list_size_cluster, file = file_txt)

print("■ 대조군 : 군집화 X", file = file_txt)
print(list_scores, file = file_txt)

print("■ 실험군 01-01 : C0 군집의 합", file = file_txt)
print(list_scores_c0, file = file_txt)
print("■ 실험군 01-02 : C1 군집의 합", file = file_txt)
print(list_scores_c1, file = file_txt)
if K == 3 : 
    print("■ 실험군 01-03 : C2 군집의 합", file = file_txt)
    print(list_scores_c2, file = file_txt)

print("■ 실험군 02 : 모든 군집의 합", file = file_txt)
print(list_scores_clustered, file = file_txt)

file_txt.close()

In [ ]:
## 논문에서 사용 보류
# ## 1차 비교 (각 실험군 02(모든 군집의 합)과 대조군(군집화 X) 예측 결과와 비교)
# str_name_figure = 'Comprison by Electric Power Usage (K' + str(K) + str_interval[1:] + ' - All of Group)' + "\n"
# plt.figure(figsize=(50, 22.5), dpi = int_dpi)
# print(str_name_figure)

# ## 대조군
# plt.plot(d_actual, color = 'black', linewidth = '7', label='Actual')
# plt.plot(model_preds, color = 'red', linewidth = '12', label='Pred', alpha=0.7)
# ## C0
# plt.plot(d_actual_clust_01, color = 'black', linewidth = '7', label='Actual_C0')
# plt.plot(model_preds_clust_01, color = 'limegreen', linewidth = '12', label='Pred_C0', alpha=0.5)
# ## C1
# plt.plot(d_actual_clust_02, color = 'black', linewidth = '7', label='Actual_C1')
# plt.plot(model_preds_clust_02, color = '#e35f62', linewidth = '12', label='Pred_C1', alpha=0.5)
# ## C2
# if K == 3 : 
#     plt.plot(d_actual_clust_03, color = 'black', linewidth = '7', label='Actual_C2')
#     plt.plot(model_preds_clust_03, color = 'blue', linewidth = '12', label='Pred_C2', alpha=0.5)

# plt.title(str_name_figure, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 90)
# plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
# plt.xticks(fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
# plt.yticks(fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
# plt.ylabel('Electric Power Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
# plt.xlim(0, 500)
# plt.legend(loc = tuple_axis_allcase, ncol = str_ncol, fontsize = 50)
# str_plt_title = 'ELEC_Comparison_' + str(dict_model[int_model]) + '_K' + str(K) + str(dict_interval[int_interval])[1:] + '_0-0_ALL Pred.png'
# # plt.savefig(str_plt_title, dpi = int_dpi)
# print(str_plt_title)

In [ ]:
## [미사용] 논문에서 사용 보류
# ## 1-1 비교 (각 실험군 02(모든 군집의 합) 비교)
# str_name_figure = 'Comprison by ' + str_domain + ' Energy Usage - (General Method)' + "\n"
# plt.figure(figsize=(50, 22.5), dpi = int_dpi)
# print(str_name_figure)

# ## 대조군
# plt.plot(d_actual, color = 'black', linewidth = '7', label='Actual')
# plt.plot(model_preds, color = 'red', linewidth = '12', label='Pred', alpha=0.5)

# plt.title(str_name_figure, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 90)
# plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
# plt.xticks(fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
# plt.yticks(fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
# plt.ylabel('Electric Power Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
# plt.xlim(0, 500)
# plt.legend(loc = (0.375, -0.2), ncol = str_ncol, fontsize = 50)
# str_plt_title = 'ELEC_Comparison_' + str(dict_model[int_model]) + '_K' + str(K) + str(dict_interval[int_interval])[1:] + '_1-0_General Pred.png'
# # plt.savefig(str_plt_title, dpi = int_dpi)
# print(str_plt_title)

In [ ]:
## C0
str_name_figure = 'Comprison by ' + str_domain + ' Energy Usage - (K' + str(K) + str_interval[1:] + ' - Group 01)' + "\n"
plt.figure(figsize=(50, 22.5), dpi = int_dpi)
print(str_name_figure)

plt.plot(model_preds_clust_01, color = 'red', linewidth = '5', label='Pred_C0')#, alpha=0.5)
plt.plot(d_actual_clust_01, color = 'black', linewidth = '5', label='Actual_C0')
plt.title(str_name_figure, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 90)
plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xticks(fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
plt.yticks(fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
plt.ylabel('Electric Power Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xlim(0, 500)
plt.legend(loc = 'upper right', ncol = 1, prop = {'family' : name_font_tnr, 'size' : 50})
str_plt_title = 'ELEC_Comparison_' + str(dict_model[int_model]) + '_K' + str(K) + str(dict_interval[int_interval])[1:] + '_1-1_C0 Pred.png'
# plt.savefig(str_plt_title, dpi = int_dpi)
print(str_plt_title)



str_name_figure = 'Comprison by ' + str_domain + ' Energy Usage - (K' + str(K) + str_interval[1:] + ' - Group 02)' + "\n"
plt.figure(figsize=(50, 22.5), dpi = int_dpi)
print(str_name_figure)

## C1
plt.plot(model_preds_clust_02, color = 'blue', linewidth = '5', label='Pred_C1')#, alpha=0.5)
plt.plot(d_actual_clust_02, color = 'black', linewidth = '5', label='Actual_C1')

plt.title(str_name_figure, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 90)
plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xticks(fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
plt.yticks(fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
plt.ylabel('Electric Power Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xlim(0, 500)
plt.legend(loc = 'upper right', ncol = 1, prop = {'family' : name_font_tnr, 'size' : 50})
str_plt_title = 'ELEC_Comparison_' + str(dict_model[int_model]) + '_K' + str(K) + str(dict_interval[int_interval])[1:] + '_1-2_C1 Pred.png'
# plt.savefig(str_plt_title, dpi = int_dpi)
print(str_plt_title)


## C2
if K == 3 : 
    str_name_figure = 'Comprison by ' + str_domain + ' Energy Usage - (K' + str(K) + str_interval[1:] + ' - Group 03)' + "\n"
    plt.figure(figsize=(50, 22.5), dpi = int_dpi)
    print(str_name_figure)

    plt.plot(d_actual_clust_03, color = 'black', linewidth = '5', label='Actual_C2')
    plt.plot(model_preds_clust_03, color = 'blue', linewidth = '5', label='Pred_C2')#, alpha=0.5)

    plt.title(str_name_figure, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 90)
    plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
    plt.xticks(fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
    plt.yticks(fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
    plt.ylabel('Electric Power Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
    plt.xlim(0, 500)
    plt.legend(loc = 'upper right', ncol = 1, prop = {'family' : name_font_tnr, 'size' : 50})
    str_plt_title = 'ELEC_Comparison_' + str(dict_model[int_model]) + '_K' + str(K) + str(dict_interval[int_interval])[1:] + '_1-3_C2 Pred.png'
    # plt.savefig(str_plt_title, dpi = int_dpi)
    print(str_plt_title)

In [ ]:
## 2차 비교 (실험군 02(모든 군집의 합)과 대조군(군집화 X) 예측 결과와 비교)
str_name_figure = 'Comprison by Electric Power Usage (K' + str(K) + str_interval[1:] + ' - Sum of Clustered Prediction)' + "\n"
plt.figure(figsize=(50, 22.5), dpi = int_dpi)
print(str_name_figure)

plt.plot(d_actual, color = 'black', linewidth = '5', label='Actual')
plt.plot(model_preds, color = 'red', linewidth = '5', label='Pred')
plt.plot(model_preds_clust, color = 'green', linewidth = '5', label='Pred_Clustered')

plt.title(str_name_figure, fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 90)
plt.xlabel('Period', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xticks(fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
plt.yticks(fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 50)
plt.ylabel('Electric Power Usage', fontproperties = FONT_TIMES_NEW_ROMAN, fontsize = 70)
plt.xlim(0, 500)
plt.legend(loc = 'upper right', ncol = 1, prop = {'family' : name_font_tnr, 'size' : 50})
str_plt_title = 'ELEC_Comparison_' + str(dict_model[int_model]) + '_K' + str(K) + str(dict_interval[int_interval])[1:] + '_2-1_Sum Pred.png'
# plt.savefig(str_plt_title, dpi = int_dpi)
print(str_plt_title)

In [ ]:
# ## 결과창 용량이 너무 커져서 모델 1~2개씩 돌리는 것으로 수정
# for int_model in range(4, 5) : ## 0:"CB", 1:"DT", 2:"LGBM", 3:"RF", 4:"XGB"
#         ## 군집화 데이터셋에 대한 별도 처리
#         for i in range (0, 10): ## 각 기간별 10번의 Clustering을 병행
#                 sys.stdout.flush() ## flush
#                 df_kier_summary_total, list_size_cluster = cluster_label()
#                 print(list_size_cluster)
#                 # df_kier_summary_total

#                 for int_grp in range(1, K + 1): ## 군집 형성된 데이터셋만 분석
#                         print('■ ' + str(int_grp))
#                         df_tar, str_col_tar = load_dataset_cluster(int_grp) ## 해당 군집에 대한 데이터셋 및 Target Column
#                         # print(df_tar.columns)
#                         # print(df_tar.shape)

#                         list_res, list_hists = com_ML.model_ml_analysis_with_KFold(df_tar, int_model, float_rate, str_col_tar, int_fold, str_shuffle = True)

#                         ## list_res 저장
#                         str_txt = '../kf_Comp_hist/kf_result_' + str(dict_interval[int_interval]) + '_K'  + str(K)  + '_Case0' + str(i) + '_' + dict_grp[int_grp] + '_' + dict_model[int_model] + '_CV' + str(int_fold) + '.txt'
#                         file_txt = open(str_txt, 'w')
#                         print('- Interval = ' + dict_interval[int_interval] + '\n'
#                                 + '- K = ' + str(K) + '\n'
#                                 + '- grp = C0' + str(int_grp) + '\n'
#                                 + '- model = ' + dict_model[int_model] + '\n'
#                                 + '- Case = 0' + str(i) + ',' + ' size_cluster = ' + str(list_size_cluster) + '\n'
#                                 + '- Size = ' + str(df_tar.shape) + '\n'
#                                 + '- Columns = ' + str(df_tar.columns) + '\n', file = file_txt)
#                         print(list_res, file = file_txt)

#                         ## list_hist 저장
#                         str_txt = '../kf_Comp_hist/kf_hist_' + str(dict_interval[int_interval]) + '_K'  + str(K)  + '_Case0' + str(i) + '_' + dict_grp[int_grp] + '_' + dict_model[int_model] + '_CV' + str(int_fold) + '.txt'
#                         file_txt = open(str_txt, 'w')
#                         print('- Interval = ' + dict_interval[int_interval] + '\n'
#                                 + '- K = ' + str(K) + '\n'
#                                 + '- grp = C0' + str(int_grp) + '\n'
#                                 + '- model = ' + dict_model[int_model] + '\n'
#                                 + '- Case = 0' + str(i) + ',' + ' size_cluster = ' + str(list_size_cluster) + '\n'
#                                 + '- Size = ' + str(df_tar.shape) + '\n'
#                                 + '- Columns = ' + str(df_tar.columns) + '\n', file = file_txt)
#                         print(list_hists, file = file_txt)

#                         ## open 후 다른 것을 open하면 자동으로 close되어 저장되지만,
#                         ## 마지막 파일은 반드시 close를 통해 종료해야만 저장이 완료됨
#                         file_txt.close()